# E2 — Hyperparameter tuning (M4)

The plan's section 2: one variable per stage, each stage starting from the previous stage's best
configuration, one seed, validation only. Stage 1 compares optimisers, stage 2 compares dropout,
stage 3 compares depth, and `final` re-runs the winner on three seeds.

Every run gets the same budget (300 epochs, no early stopping, batch 512) so the stages are
comparable and match E1 (`e1_stress_results_new_transformer`).
The test set is never loaded.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0), "|", torch.__version__)

In [ ]:
import os, subprocess, sys
from pathlib import Path

BRANCH = "baselines"

if not Path('/content/repo/.git').exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/RayYouDS/cits4012-NLP-project.git",
                    "/content/repo"], check=True)
os.chdir('/content/repo')

# A clone left over from an earlier session can sit on a different branch, or on an older commit
# of the right branch. The branch name alone does not tell you which code is about to run.
on = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], text=True).strip()
if on != BRANCH:
    raise SystemExit(f"/content/repo is on {on}, not {BRANCH}. Delete /content/repo and rerun.")

subprocess.run(['git', 'fetch', '-q', 'origin', BRANCH], check=True)
local = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], text=True).strip()
if local != remote:
    behind = subprocess.check_output(
        ['git', 'rev-list', '--count', f'{local}..{remote}'], text=True).strip()
    raise SystemExit(
        f"/content/repo is at {local[:9]} but origin/{BRANCH} is at {remote[:9]} "
        f"({behind} commits ahead). Delete /content/repo and rerun so the run uses current code.")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece'], check=True)

for required in ["scripts/training/run_m3.py",
                 "scripts/tokenizer/piqa_bpe_v3.model", "datasets/PIQA/train.jsonl"]:
    if not Path(required).is_file():
        raise SystemExit(f"{required} missing - clone failed or the branch moved.")

# Stage 2 compares dropout. Without the wiring every run silently uses 0.1 and the stage
# would compare three identical models.
src = Path('scripts/training/run_m3.py').read_text(encoding='utf-8')
builder = src[src.index('def build_model'):src.index('def build_optimizer')]
missing = [k for k in ('dropout_rate', 'pooling', 'enable_pos_emb') if k not in builder]
if missing:
    raise SystemExit(f"build_model does not pass {missing} - this branch predates that wiring.")

# run_m3 seeds a fresh generator per run, so different seeds do get different batch orders.
# What preloading removes is the per-epoch reshuffle within a run: the bucket sampler runs once
# and every epoch then repeats the same batches in the same order.
if 'preload_dataloader_to_gpu(\n                    train_loader' in src:
    print("NOTE: the training loader is preloaded into a fixed list, so within each run every"
          " epoch sees the same batches in the same order. Seeds still differ from each other.")

# The commit only describes what runs if nobody edited the checkout. -uno looks at tracked
# files only, so results written elsewhere and __pycache__ cannot raise a false alarm.
edited = subprocess.check_output(['git', 'status', '--porcelain', '-uno'], text=True).strip()
if edited:
    raise SystemExit(f"/content/repo has local edits:\n{edited}\nDelete /content/repo and rerun.")

COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('branch:', BRANCH, ' commit:', COMMIT[:9], ' clean working tree')


## Where results go

A run is saved only once it finishes; a disconnect loses the run in flight and there is no resume.
Later stages read earlier stages' results off disk, so losing them means starting over.


In [ ]:
STAGE      = 'final'      # 1, 2, 3, or "final"
USE_DRIVE  = True     # False -> results live in /content and die with the runtime
SMOKE_TEST = False    # True -> 3 epochs per run, into a separate directory

from pathlib import Path
RUN_TAG = 'e2_smoke_300' if SMOKE_TEST else 'e2_full_300'   # separate from the 150-epoch run
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = Path('/content/drive/MyDrive/cits4012') / RUN_TAG
else:
    OUT_ROOT = Path('/content') / RUN_TAG
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("results root ->", OUT_ROOT)


## Configuration

Each run's config is copied next to its results as `config.json`, so a later stage reads the
winning configuration from disk instead of having it typed in again.


In [ ]:
import json, shutil
import pandas as pd

# Embedded so this runs against the shared branch with nothing pushed first.
BASE = {   'output_dir': 'reports/e2_base',
    'tokenizer': 'scripts/tokenizer/piqa_bpe_v3.model',
    'models': ['transformer'],
    'learning_rates': [0.0003],
    'seeds': [4012],
    'optimizer': 'adam',
    'weight_decay': None,
    'momentum': 0.9,
    'batch_size': 512,
    'max_epochs': 300,
    'patience': None,
    'grad_clip': None,
    'num_workers': 2,
    'embedding_dim': 100,
    'hidden_size': 100,
    'bidirectional': True,
    'enable_attention': True,
    'num_heads': 4,
    'num_layers': 3,
    'dropout_rate': 0.1,
    'pooling': 'cls',
    'enable_pos_emb': True}


def finished(prefixes):
    """(config, summary row) for every finished run whose directory starts with any prefix."""
    if isinstance(prefixes, str):
        prefixes = (prefixes,)
    out = []
    seen = set()
    for d in sorted(d for prefix in prefixes for d in OUT_ROOT.glob(prefix + '*')):
        if d.name in seen:
            continue
        seen.add(d.name)
        summary, config = d / 'm3_summary.csv', d / 'config.json'
        if summary.is_file() and config.is_file():
            cfg = json.load(open(config, encoding='utf-8'))
            for _, row in pd.read_csv(summary).iterrows():
                out.append((d.name, cfg, row))
    return out


# best() ranks by validation accuracy. When McNemar says the top two are a tie, the plan's rule
# is to pick by overfitting gap, then time to best, then parameter count - which best() cannot do
# for you. Record that choice here, keyed by the stage it applies to, rather than deleting
# directories to force the outcome. The chosen directory is written into the next stage's config
# as _from, so the decision stays visible afterwards.
PICK = {
    # Set these only after reading the stage's ranking. Leave empty to take the highest
    # validation accuracy. For the 300-epoch rerun, stage 1 must finish before PICK is filled in.
}


def best(prefixes):
    runs = finished(prefixes)
    if not runs:
        raise SystemExit(f"no finished results under {OUT_ROOT}/{prefixes} - run that stage first.")
    ranked = sorted(runs, key=lambda r: r[2].valid_accuracy, reverse=True)
    chosen = PICK.get(STAGE)
    if chosen:
        match = [r for r in ranked if r[0] == chosen]
        if not match:
            raise SystemExit(f"PICK[{STAGE}] = {chosen!r} is not a finished run under {prefixes}. "
                             f"Available: {[r[0] for r in ranked]}")
        if match[0] is not ranked[0]:
            print(f"PICK[{STAGE}]: using {chosen} instead of the highest validation accuracy "
                  f"({ranked[0][0]}, {ranked[0][2].valid_accuracy:.4f}).")
        ranked = match + [r for r in ranked if r[0] != chosen]
    name, cfg, row = ranked[0]
    print(f"best of {prefixes}: {name}  {cfg['optimizer']} lr={row.learning_rate:g} "
          f"dropout={cfg['dropout_rate']} layers={cfg['num_layers']} -> valid {row.valid_accuracy:.4f}")
    # The margin matters here, not three cells later: this winner is what the next stage builds on.
    if len(ranked) > 1:
        margin = row.valid_accuracy - ranked[1][2].valid_accuracy
        runner = ranked[1]
        print(f"  runner-up {runner[0]} at {runner[2].valid_accuracy:.4f}, margin {margin*100:.2f} pp")
        if margin < 0.0125:
            print("  margin is small. The results cell runs McNemar on these two; if that comes out"
                  " non-significant, the plan's E1 rule says to prefer the smaller overfitting gap,"
                  " then the shorter time, then fewer parameters, and to record the choice.")
    # The learning rate comes from the row (a stage sweeps several); everything else from its config.
    keep = {k: cfg[k] for k in ('optimizer', 'weight_decay', 'momentum',
                                'dropout_rate', 'num_layers', 'num_heads')}
    keep['learning_rates'] = [float(row.learning_rate)]
    keep['_from'] = name
    return keep


def job(name, **overrides):
    cfg = dict(BASE)
    cfg['seeds'] = [4012]
    cfg['output_dir'] = str(OUT_ROOT / name)
    cfg.update(overrides)
    if SMOKE_TEST:
        cfg['max_epochs'] = 3
    return name, cfg


if STAGE == 1:
    # The plan's grid: E1's best learning rate divided by 3, itself, and times 3.
    lr = BASE['learning_rates'][0]
    GRID = [float(f'{x:.2g}') for x in (lr / 3, lr, lr * 3)]
    print('learning rates:', GRID)
    # One learning rate per invocation. run_m3 writes m3_summary.csv only after every run in
    # a config finishes, so a disconnect part way through a three-rate config throws away the
    # rates that already finished. One rate per directory costs a little startup and keeps them.
    JOBS = []
    for optimizer, grid, decay in (('adam', GRID, None),
                                   ('adamw', GRID, 0.01),
                                   ('sgd', [0.01, 0.03, 0.1], None)):
        for rate in grid:
            JOBS.append(job(f's1_{optimizer}_lr{rate:g}', optimizer=optimizer,
                            learning_rates=[rate], weight_decay=decay))

elif STAGE == 2:
    keep = best('s1_')
    # 0.1 is already covered by stage 1.
    JOBS = [job(f's2_dropout{d}', **{**keep, 'dropout_rate': d}) for d in (0.0, 0.3)]

elif STAGE == 3:
    # Stage 2 only ran dropout 0.0 and 0.3 because 0.1 is the stage 1 winner. Looking at s2_ alone
    # would throw that candidate away, so stage 3 starts from the best of stage 1 and stage 2.
    keep = best(('s1_', 's2_'))
    # Depth, not head count: head count leaves the parameter total unchanged, so the plan's
    # "fewer parameters" tie-break cannot separate 2 heads from 4.
    JOBS = [job('s3_layers2', **{**keep, 'num_layers': 2})]

elif STAGE == 'final':
    keep = best(('s1_', 's2_', 's3_'))
    name, cfg = job('final_3seed', **keep)
    cfg['seeds'] = [4012, 4013, 4014]
    JOBS = [(name, cfg)]

else:
    raise SystemExit(f"STAGE must be 1, 2, 3 or 'final', got {STAGE!r}")


PLAN = []
UNVERIFIED = []
for name, cfg in JOBS:
    cfg['_commit'] = COMMIT
    out = Path(cfg['output_dir'])
    if (out / 'm3_summary.csv').is_file():
        # Reusing a finished directory is only safe if it was produced by this exact plan.
        old = json.load(open(out / 'config.json', encoding='utf-8'))
        # Underscore keys are metadata. A key absent from one side falls back to the model's
        # default, so writing pooling='cls' explicitly does not invalidate an earlier run that
        # relied on the same default.
        defaults = {'pooling': 'cls', 'enable_pos_emb': True,
                    'include_goal': True, 'dropout_rate': 0.1}
        differs = sorted(k for k in set(old) | set(cfg)
                         if not k.startswith('_') and k != 'output_dir'
                         and old.get(k, defaults.get(k)) != cfg.get(k, defaults.get(k)))
        if differs:
            raise SystemExit(f"{out} is finished but its config differs on {differs}. "
                             f"Move or delete that directory, or revert the change.")
        old_commit = old.get('_commit')
        if old_commit is None:
            UNVERIFIED.append(name)
        elif old_commit != COMMIT:
            # A different commit only matters if it changed code this experiment runs. Baseline
            # scripts and documentation move independently and must not invalidate results.
            touched = subprocess.check_output(
                ['git', 'diff', '--name-only', f'{old_commit}..{COMMIT}', '--',
                 'scripts/training/', 'scripts/models/', 'scripts/data_loader/',
                 'scripts/evaluation/', 'scripts/tokenizer/', 'datasets/'],
                text=True).split()
            if touched:
                raise SystemExit(
                    f"{out} was produced at {old_commit[:9]}, and {COMMIT[:9]} changes code it "
                    f"depends on: {touched}. Move or delete that directory and rerun.")
            print(f"  {name}: produced at {old_commit[:9]}, reused - the difference to "
                  f"{COMMIT[:9]} is outside the code this experiment runs")
        print(f"skip {name} - already finished")
        continue
    # config.json is written here before the run starts, so a directory holding only that has
    # not been used yet and re-running this cell is harmless. Anything else without a summary
    # is a run that died part way: skipping it would hide the gap, reusing the directory would
    # mix two attempts.
    leftovers = sorted(p.name for p in out.glob('*') if p.name != 'config.json') if out.exists() else []
    if leftovers:
        raise SystemExit(f"{out} holds {leftovers[:4]} but no m3_summary.csv - a run died there. "
                         f"Delete that directory, then rerun this cell.")
    out.mkdir(parents=True, exist_ok=True)
    path = f'/content/e2_{name}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(cfg, f, indent=2)
    shutil.copy(path, out / 'config.json')
    PLAN.append((name, path, cfg))
    print(f"  {name:16s} {cfg['optimizer']:6s} lr={cfg['learning_rates']} "
          f"dropout={cfg['dropout_rate']} layers={cfg['num_layers']} seeds={cfg['seeds']}")

if UNVERIFIED:
    print(f"\nWARNING: {UNVERIFIED} carry no _commit, so the code that produced them cannot be"
          f" checked against {COMMIT[:9]}. They were written before this notebook started"
          f" recording it. Delete and rerun them if you want that guarantee.")

n_runs = sum(len(c['learning_rates']) * len(c['seeds']) for _, _, c in PLAN)
print(f"\nstage {STAGE}: {n_runs} runs, up to {n_runs * 12.75:.0f} min on a T4"
      f" (300 epochs at 2.55 s/epoch; no early stopping)")


## Run

In [ ]:
import time

for name, path, cfg in PLAN:
    print('=' * 72); print(name); print('=' * 72)
    started = time.time()
    proc = subprocess.Popen(
        [sys.executable, '-u', '-m', 'scripts.training.run_m3', '--config', path],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode:
        raise SystemExit(f"{name} exited with code {proc.returncode}")
    print(f"{name} done in {(time.time() - started) / 60:.1f} min")


## Results so far

In [ ]:
import math

runs = finished('')
if not runs:
    print("nothing finished yet")
else:
    rows = []
    for name, cfg, row in runs:
        # valid_accuracy is the best of 100 noisy epochs and overshoots the plateau.
        # plateau_mean averages the second half of the same curve as a second opinion.
        # Early stopping makes runs different lengths, so it is a fraction, not a fixed epoch.
        run_id = f"m3_{row.model}_lr{row.learning_rate:g}_seed{int(row.seed)}"
        hist = OUT_ROOT / name / f"{run_id}_history.csv"
        if hist.is_file():
            curve = pd.read_csv(hist).valid_accuracy
            plateau = round(float(curve.iloc[len(curve) // 2:].mean()), 4)
        else:
            plateau = None
        rows.append({'_pred': str(OUT_ROOT / name / f"{run_id}_val_predictions.csv"),
                     'stage': name, 'optimizer': cfg['optimizer'],
                     'lr': row.learning_rate, 'weight_decay': cfg['weight_decay'],
                     'dropout': cfg['dropout_rate'], 'layers': cfg['num_layers'],
                     'heads': cfg['num_heads'], 'seed': int(row.seed),
                     'best_epoch': int(row.best_epoch), 'epochs_run': int(row.epochs_run),
                     'valid': round(row.valid_accuracy, 4),
                     'plateau_mean': plateau, 'gap': round(row.gap_at_best, 3),
                     'params': int(row.params)})
    table = pd.DataFrame(rows).sort_values('valid', ascending=False).reset_index(drop=True)
    display(table.drop(columns=['_pred']))

    # sqrt(0.25/1606) = 1.25 pp is the standard error of a single accuracy, not of the paired
    # difference between two models scored on the same 1606 questions. It is a rough yardstick
    # only. McNemar on the two prediction files is the test the plan actually specifies.
    print(f"Rough yardstick: 1 s.e. on a single accuracy = {math.sqrt(0.25/1606)*100:.2f} pp "
          f"(n = 1606). Not a paired test.")

    if len(table) > 1:
        top, second = table.iloc[0], table.iloc[1]
        print(f"Top two differ by {(top.valid - second.valid)*100:.2f} pp "
              f"({top.stage} vs {second.stage}).")
        try:
            from scripts.evaluation.evaluate import mcnemar_test
            a = pd.read_csv(top._pred)[['row_id', 'correct']]
            b = pd.read_csv(second._pred)[['row_id', 'correct']]
            merged = a.merge(b, on='row_id', suffixes=('_a', '_b'))
            result = mcnemar_test(merged.correct_a.to_numpy().astype(bool),
                                  merged.correct_b.to_numpy().astype(bool))
            verdict = 'significant' if result['p_value'] < 0.05 else 'NOT significant'
            print(f"McNemar on the same {len(merged)} questions: only_a={result['only_a']} "
                  f"only_b={result['only_b']} chi2={result['chi2']:.3f} p={result['p_value']:.4f}"
                  f"  -> {verdict} at alpha 0.05")
            if result['p_value'] >= 0.05:
                print("The plan's E1 rule for a tie: prefer the smaller overfitting gap, then the"
                      " shorter time to best, then fewer parameters. Record which one you applied.")
        except Exception as exc:
            print(f"McNemar skipped: {exc}")
    if table.plateau_mean.notna().any() and table.sort_values(
            'plateau_mean', ascending=False).index[0] != 0:
        print("best_epoch and plateau_mean disagree about the winner - this stage separated nothing.")
